```
  set Density (rho)                           = 1
  set Elastic modulus of the vessel wall (mu) = 1
  set Reference cross-sectional area (a0)     = 1
  set Reference pressure (p0)                 = 1
  set Reference radius (r0)                   = 1
  set Tube law exponent (m)                   = 0.5
  set Viscosity coefficient (eta_c)           = 1
```

In [34]:
import sympy
from sympy import *
rho, mu, a0, p0, r0, m, eta_c, x, t = symbols('rho mu a_0 p_0 r_0 m eta_c x t')

A = Function('A')(x, t)
U = Function('U')(x, t)

dA = Function('\delta A')(x, t)
dU = Function('\delta U')(x, t)

fA = Function('f_A')(x, t)
fU = Function('f_U')(x, t)

# pressure
P = p0 + mu*( (A/a0)**m - 1 )

# fluxes
flux_A = A*U
flux_U = U**2/2 + P/rho

# manufactured RHS
fA_man = simplify(diff(A, t) + diff(flux_A, x) )
fU_man = simplify(diff(U, t) + diff(flux_U, x) + eta_c * U)

In [35]:
# implicit function for ARKODE
LA = fA - diff(flux_A, x)
LU = fU - diff(flux_U, x) - eta_c * U

L = Matrix([LA, LU])
W = Matrix([A, U])
dW = Matrix([dA, dU])

# Frechet derivative of L in the direction dW (symbolic expression of J*dW)
eps = symbols('epsilon')
JdW = (L.subs({A: A + eps*dA, U: U + eps * dU}) - L).diff(eps).subs({eps: 0})

In [36]:
JdW

Matrix([
[                                                                                                                                                                            -A(x, t)*Derivative(\delta U(x, t), x) - U(x, t)*Derivative(\delta A(x, t), x) - \delta A(x, t)*Derivative(U(x, t), x) - \delta U(x, t)*Derivative(A(x, t), x)],
[-eta_c*\delta U(x, t) - m**2*mu*(A(x, t)/a_0)**m*\delta A(x, t)*Derivative(A(x, t), x)/(rho*A(x, t)**2) - m*mu*(A(x, t)/a_0)**m*Derivative(\delta A(x, t), x)/(rho*A(x, t)) + m*mu*(A(x, t)/a_0)**m*\delta A(x, t)*Derivative(A(x, t), x)/(rho*A(x, t)**2) - U(x, t)*Derivative(\delta U(x, t), x) - \delta U(x, t)*Derivative(U(x, t), x)]])

In [37]:
Aexp = a0*(1+.5*sin(2*pi*x-t*pi))
Aexp = x+1
Aexp = 0*x+1
Uexp = 0*x+1

dAexp = 0*x+1
dUexp = 0*x+1

A0exp = Aexp.subs({t:0})
U0exp = Uexp.subs({t:0})

s = {A: Aexp, U: Uexp,
     dA: dAexp, dU: dUexp}

fA_man_exp = fA_man.subs(s).simplify()
fU_man_exp = fU_man.subs(s).simplify()

Lexp = L.subs(s).subs({fA:fA_man_exp, fU:fU_man_exp})

JdWexp = JdW.subs(s)

In [38]:
s = f"""
   set Exact solution = {Aexp}; {Uexp}
   set Initial condition = {A0exp}; {U0exp}
   set RHS expression = {fA_man_exp}; {fU_man_exp}

   set L(W) = {Lexp[0].simplify()}; {Lexp[1].simplify()}
   set dW = {dAexp}; {dUexp}
   set J*dW = {JdWexp[0,0].simplify()}; {JdWexp[1,0].simplify()}
"""

print(s.replace('pi', 'PI').replace('**', '^'))


   set Exact solution = 1; 1
   set Initial condition = 1; 1
   set RHS expression = 0; eta_c

   set L(W) = 0; 0
   set dW = 1; 1
   set J*dW = 0; -eta_c



In [39]:
# HLL residual flux translated from blood_flow_system.h
A_L_sym, U_L_sym, A_R_sym, U_R_sym, bn_L, bn_R = symbols('A_L U_L A_R U_R bn_L bn_R')

def pressure(area):
    return p0 + mu*((area/a0)**m - 1)

def wave_speed(area):
    dpdA = diff(pressure(area), area)
    return sqrt(area/rho * dpdA)

c_L = wave_speed(A_L_sym)
c_R = wave_speed(A_R_sym)
U_bar = (U_L_sym + U_R_sym)/2
c_bar = (c_L + c_R)/2
s_L = U_bar - c_bar
s_R = U_bar + c_bar

# physical fluxes projected on the normal direction (b · n = bn)
FAL = A_L_sym * U_L_sym * bn_L
FUL = (0.5 * U_L_sym**2 + pressure(A_L_sym)/rho) * bn_L

FAR = A_R_sym * U_R_sym * bn_R
FUR = (0.5 * U_R_sym**2 + pressure(A_R_sym)/rho) * bn_R

FHLL_A = Piecewise(
    (FAL, s_L >= 0),
    (FAR, s_R <= 0),
    ((s_R*FAL - s_L*FAR + s_R*s_L*(A_R_sym - A_L_sym))/(s_R - s_L), True)
)

FHLL_U = Piecewise(
    (FUL, s_L >= 0),
    (FUR, s_R <= 0),
    ((s_R*FUL - s_L*FUR + s_R*s_L*(U_R_sym - U_L_sym))/(s_R - s_L), True)
)

FHLL_A, FHLL_U


(Piecewise((A_L*U_L*bn_L, U_L/2 + U_R/2 - sqrt(m*mu*(A_L/a_0)**m/rho)/2 - sqrt(m*mu*(A_R/a_0)**m/rho)/2 >= 0), (A_R*U_R*bn_R, U_L/2 + U_R/2 + sqrt(m*mu*(A_L/a_0)**m/rho)/2 + sqrt(m*mu*(A_R/a_0)**m/rho)/2 <= 0), ((A_L*U_L*bn_L*(U_L/2 + U_R/2 + sqrt(m*mu*(A_L/a_0)**m/rho)/2 + sqrt(m*mu*(A_R/a_0)**m/rho)/2) - A_R*U_R*bn_R*(U_L/2 + U_R/2 - sqrt(m*mu*(A_L/a_0)**m/rho)/2 - sqrt(m*mu*(A_R/a_0)**m/rho)/2) + (-A_L + A_R)*(U_L/2 + U_R/2 - sqrt(m*mu*(A_L/a_0)**m/rho)/2 - sqrt(m*mu*(A_R/a_0)**m/rho)/2)*(U_L/2 + U_R/2 + sqrt(m*mu*(A_L/a_0)**m/rho)/2 + sqrt(m*mu*(A_R/a_0)**m/rho)/2))/(sqrt(m*mu*(A_L/a_0)**m/rho) + sqrt(m*mu*(A_R/a_0)**m/rho)), True)),
 Piecewise((bn_L*(0.5*U_L**2 + (mu*((A_L/a_0)**m - 1) + p_0)/rho), U_L/2 + U_R/2 - sqrt(m*mu*(A_L/a_0)**m/rho)/2 - sqrt(m*mu*(A_R/a_0)**m/rho)/2 >= 0), (bn_R*(0.5*U_R**2 + (mu*((A_R/a_0)**m - 1) + p_0)/rho), U_L/2 + U_R/2 + sqrt(m*mu*(A_L/a_0)**m/rho)/2 + sqrt(m*mu*(A_R/a_0)**m/rho)/2 <= 0), ((bn_L*(0.5*U_L**2 + (mu*((A_L/a_0)**m - 1) + p_0)/rho)*(U_L/

In [40]:
# Frechet derivative of HLL flux in directions dW_L and dW_R
dA_L, dU_L, dA_R, dU_R = symbols('\delta~A_L \delta~U_L \delta~A_R \delta~U_R')
dA_L, dU_L, dA_R, dU_R = symbols('trial_A_L trial_U_L trial_A_R trial_U_R')
eps = symbols('epsilon')

subs_eps = {
    A_L_sym: A_L_sym + eps*dA_L,
    U_L_sym: U_L_sym + eps*dU_L,
    A_R_sym: A_R_sym + eps*dA_R,
    U_R_sym: U_R_sym + eps*dU_R,
}

FHLL_A_eps = FHLL_A.subs(subs_eps)
FHLL_U_eps = FHLL_U.subs(subs_eps)

dFHLL_A = diff(FHLL_A_eps, eps).subs(eps, 0)
dFHLL_U = diff(FHLL_U_eps, eps).subs(eps, 0)

dFHLL_A, dFHLL_U


(Piecewise((A_L*bn_L*trial_U_L + U_L*bn_L*trial_A_L, U_L/2 + U_R/2 - sqrt(m*mu*(A_L/a_0)**m/rho)/2 - sqrt(m*mu*(A_R/a_0)**m/rho)/2 >= 0), (A_R*bn_R*trial_U_R + U_R*bn_R*trial_A_R, U_L/2 + U_R/2 + sqrt(m*mu*(A_L/a_0)**m/rho)/2 + sqrt(m*mu*(A_R/a_0)**m/rho)/2 <= 0), ((-m*trial_A_R*sqrt(m*mu*(A_R/a_0)**m/rho)/(2*A_R) - m*trial_A_L*sqrt(m*mu*(A_L/a_0)**m/rho)/(2*A_L))*(A_L*U_L*bn_L*(U_L/2 + U_R/2 + sqrt(m*mu*(A_L/a_0)**m/rho)/2 + sqrt(m*mu*(A_R/a_0)**m/rho)/2) - A_R*U_R*bn_R*(U_L/2 + U_R/2 - sqrt(m*mu*(A_L/a_0)**m/rho)/2 - sqrt(m*mu*(A_R/a_0)**m/rho)/2) + (-A_L + A_R)*(U_L/2 + U_R/2 - sqrt(m*mu*(A_L/a_0)**m/rho)/2 - sqrt(m*mu*(A_R/a_0)**m/rho)/2)*(U_L/2 + U_R/2 + sqrt(m*mu*(A_L/a_0)**m/rho)/2 + sqrt(m*mu*(A_R/a_0)**m/rho)/2))/(sqrt(m*mu*(A_L/a_0)**m/rho) + sqrt(m*mu*(A_R/a_0)**m/rho))**2 + (A_L*U_L*bn_L*(trial_U_L/2 + trial_U_R/2 + m*trial_A_R*sqrt(m*mu*(A_R/a_0)**m/rho)/(4*A_R) + m*trial_A_L*sqrt(m*mu*(A_L/a_0)**m/rho)/(4*A_L)) + A_L*bn_L*trial_U_L*(U_L/2 + U_R/2 + sqrt(m*mu*(A_L/a_0)**m/

In [42]:
# C++ code helpers for dFHLL_A and dFHLL_U
from sympy import ccode, cse, numbered_symbols

def ccode_with_defs(expr, name):
    temps, (main,) = cse(expr, symbols=numbered_symbols('t_'))

    def qualify_std(line):
        line = line.replace('pow(', 'std::pow(')
        line = line.replace('sqrt(', 'std::sqrt(')
        return line

    lines = [f'// {name}']
    lines += [qualify_std(f'const double {sym} = {ccode(val)};') for sym, val in temps]
    lines += [qualify_std(f'const double {name} = {ccode(main)};')]
    return "\n".join(lines)

print(ccode_with_defs(FHLL_A, 'FHLL_A'))
print()
print(ccode_with_defs(FHLL_U, 'FHLL_U'))
print()
print(ccode_with_defs(dFHLL_A, 'dFHLL_A'))
print()
print(ccode_with_defs(dFHLL_U, 'dFHLL_U'))


// FHLL_A
const double t_0 = A_L*U_L*bn_L;
const double t_1 = 1.0/a_0;
const double t_2 = m*mu/rho;
const double t_3 = std::sqrt(t_2*std::pow(A_L*t_1, m));
const double t_4 = (1.0/2.0)*t_3;
const double t_5 = std::sqrt(t_2*std::pow(A_R*t_1, m));
const double t_6 = (1.0/2.0)*t_5;
const double t_7 = (1.0/2.0)*U_L + (1.0/2.0)*U_R;
const double t_8 = -t_4 - t_6 + t_7;
const double t_9 = A_R*U_R*bn_R;
const double t_10 = t_4 + t_6 + t_7;
const double FHLL_A = ((t_8 >= 0) ? (
   t_0
)
: ((t_10 <= 0) ? (
   t_9
)
: (
   (t_0*t_10 + t_10*t_8*(-A_L + A_R) - t_8*t_9)/(t_3 + t_5)
)));

// FHLL_U
const double t_0 = 1.0/rho;
const double t_1 = 1.0/a_0;
const double t_2 = std::pow(A_L*t_1, m);
const double t_3 = bn_L*(0.5*std::pow(U_L, 2) + t_0*(mu*(t_2 - 1) + p_0));
const double t_4 = m*mu*t_0;
const double t_5 = std::sqrt(t_2*t_4);
const double t_6 = (1.0/2.0)*t_5;
const double t_7 = std::pow(A_R*t_1, m);
const double t_8 = std::sqrt(t_4*t_7);
const double t_9 = (1.0/2.0)*t_8;
const double t_10 = 